# 用 BaiduNetdisk 汇算匹配桥检查 full_data 2017 企业覆盖率

这个 notebook 单独检查：如果不用当前项目里的 `cid_entid_unique.dta`，而是改用学长代码里的 `H:\\BaiduNetdiskDownload\\汇算file\\final_joinby_matched_data_2017_With_cid.dta`，那么 `full_data.dta` 中 2017 年企业能匹配上多少。

核心口径：

1. 从 `full_data.dta` 中抽取 2017 年唯一 `firm_id`；
2. 把 `firm_id` 转成数值型 `cid`；
3. 分块读取 BaiduNetdisk 的 `final_joinby_matched_data_2017_With_cid.dta`；
4. 按 `cid` 去重，保留第一条；
5. 用 `cid` 和 `full_data` 2017 企业左连接；
6. 分母始终是 `full_data` 2017 年唯一企业数。

注意：BaiduNetdisk 这个 dta 文件大约 6GB，所以也用分块读取。


## 0. 设置路径和参数


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

ROOT = Path(r'G:\Kuangyu_Temp\Outsource')
PROJECT = ROOT / 'productivity'
PROJECT_INNER = PROJECT / 'productivity'

FULL_DATA = ROOT / 'full_data.dta'
BAIDU_MATCH = Path(r'H:\BaiduNetdiskDownload\汇算file\final_joinby_matched_data_2017_With_cid.dta')
OUT_CSV = PROJECT_INNER / 'full_data_2017_baidu_huisuan_match_check.csv'
OUT_DTA = PROJECT_INNER / 'full_data_2017_baidu_huisuan_match_check.dta'

CHUNKSIZE_FULL = 500_000
CHUNKSIZE_BAIDU = 500_000

print('FULL_DATA:', FULL_DATA, '| exists =', FULL_DATA.exists())
print('BAIDU_MATCH:', BAIDU_MATCH, '| exists =', BAIDU_MATCH.exists())
print('OUT_CSV:', OUT_CSV)
print('OUT_DTA:', OUT_DTA)


## 1. 查看 BaiduNetdisk 匹配桥的变量名

先不读完整数据，只读取 metadata，确认里面有哪些变量。重点看有没有 `cid`、`id`、`eid`。


In [ ]:
baidu_meta = pd.read_stata(
    BAIDU_MATCH,
    iterator=True,
    convert_categoricals=False,
)

print(type(baidu_meta))
print([x for x in dir(baidu_meta) if 'var' in x.lower() or 'col' in x.lower() or 'name' in x.lower()])

baidu_sample = pd.read_stata(
    BAIDU_MATCH,
    chunksize=5,
    convert_categoricals=False,
)
baidu_sample = next(baidu_sample)

baidu_columns = list(baidu_sample.columns)
print('number of variables:', len(baidu_columns))
print('number of observations from metadata:', baidu_meta.nobs)
print('first 5 rows:')
display(baidu_sample.head())

baidu_columns


## 2. 决定需要读取的 BaiduNetdisk 桥接变量

这里尽量少读变量，降低内存压力。只要 `cid` 一定要读；如果有 `id`、`eid`、`match_step` 等变量，也一起保留，方便后面检查。


In [ ]:
wanted_cols = [
    'cid', 'id', 'eid',
    'company_name', 'match_step', 'one', 'a',
    'operating_revenue_max',
]

baidu_read_cols = []
for col in wanted_cols:
    if col in baidu_columns:
        baidu_read_cols.append(col)

print('columns to read from BAIDU_MATCH:')
print(baidu_read_cols)

if 'cid' not in baidu_read_cols:
    raise ValueError('BaiduNetdisk matched file does not contain cid. Need to inspect variable names manually.')


## 3. 从 full_data.dta 中提取 2017 年唯一企业

这一步和之前 notebook 一样：分块读取 `full_data.dta`，只取 `firm_id` 和 `year` 两列，然后保留 2017 年唯一企业。


In [ ]:
firm_ids = set()
rows_seen = 0
rows_2017_seen = 0

reader = pd.read_stata(
    FULL_DATA,
    columns=['firm_id', 'year'],
    chunksize=CHUNKSIZE_FULL,
    convert_categoricals=False,
)

for i, chunk in enumerate(reader, start=1):
    rows_seen = rows_seen + len(chunk)
    chunk_2017 = chunk.loc[chunk['year'] == 2017, ['firm_id']].copy()
    rows_2017_seen = rows_2017_seen + len(chunk_2017)

    chunk_2017['firm_id'] = (
        chunk_2017['firm_id']
        .astype('string')
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
    )

    firm_ids.update(chunk_2017['firm_id'].dropna().unique().tolist())

    print(
        f'chunk {i:>3}: total rows seen = {rows_seen:,}, '
        f'2017 rows seen = {rows_2017_seen:,}, '
        f'unique firm_id so far = {len(firm_ids):,}'
    )


## 4. 生成 full_data 2017 唯一企业表，并转成 cid


In [ ]:
full2017 = pd.DataFrame({'firm_id': sorted(firm_ids)})
full2017['cid'] = pd.to_numeric(full2017['firm_id'], errors='coerce')

n_full = len(full2017)
n_missing_cid = int(full2017['cid'].isna().sum())

print('full_data 2017 unique firms:', f'{n_full:,}')
print('missing cid after converting firm_id to numeric:', f'{n_missing_cid:,}')
print('missing cid rate:', f'{n_missing_cid / n_full:.2%}')

full2017.head()


## 5. 准备 full_data 的 cid 集合

后面读取 BaiduNetdisk 大文件时，只保留 `cid` 出现在 full_data 2017 企业里的行。这样可以明显减少内存占用。


In [ ]:
full_cids = set(full2017['cid'].dropna().astype('int64').tolist())

print('nonmissing full_data 2017 cid count:', f'{len(full_cids):,}')
print('first 10 cids:', list(full_cids)[:10])


## 6. 分块读取 BaiduNetdisk 匹配桥，并只保留 full_data 2017 里的 cid

这一步可能比较慢，因为 `final_joinby_matched_data_2017_With_cid.dta` 大约 6GB。

每个 chunk 做四件事：

1. 把 `cid` 转成数值；
2. 删除 `cid` 缺失的行；
3. 只保留 `cid` 属于 full_data 2017 企业的行；
4. 暂存这些候选匹配记录。


In [ ]:
matched_chunks = []
baidu_rows_seen = 0
baidu_rows_with_cid = 0
baidu_rows_in_full = 0

baidu_reader = pd.read_stata(
    BAIDU_MATCH,
    columns=baidu_read_cols,
    chunksize=CHUNKSIZE_BAIDU,
    convert_categoricals=False,
)

for i, chunk in enumerate(baidu_reader, start=1):
    baidu_rows_seen = baidu_rows_seen + len(chunk)

    chunk['cid'] = pd.to_numeric(chunk['cid'], errors='coerce')
    chunk = chunk.loc[chunk['cid'].notna()].copy()
    baidu_rows_with_cid = baidu_rows_with_cid + len(chunk)

    chunk['cid_int'] = chunk['cid'].astype('int64')
    chunk = chunk.loc[chunk['cid_int'].isin(full_cids)].copy()
    baidu_rows_in_full = baidu_rows_in_full + len(chunk)

    if len(chunk) > 0:
        matched_chunks.append(chunk)

    print(
        f'baidu chunk {i:>3}: rows seen = {baidu_rows_seen:,}, '
        f'rows with cid = {baidu_rows_with_cid:,}, '
        f'rows whose cid is in full_data 2017 = {baidu_rows_in_full:,}, '
        f'stored chunks = {len(matched_chunks):,}'
    )


## 7. 合并暂存结果，并按 cid 去重

学长代码里通常会 `bysort cid: keep if _n == 1` 或 `bysort cid year: keep if _n == 1`。

这里是 2017 年单年文件，所以按 `cid` 去重，保留第一条。


In [ ]:
if len(matched_chunks) == 0:
    baidu_in_full = pd.DataFrame(columns=baidu_read_cols + ['cid_int'])
else:
    baidu_in_full = pd.concat(matched_chunks, ignore_index=True)

print('Baidu rows matched to full_data cid before dedup:', f'{len(baidu_in_full):,}')
print('unique matched cid before dedup:', f"{baidu_in_full['cid_int'].nunique():,}")

baidu_unique = baidu_in_full.drop_duplicates(subset=['cid_int'], keep='first').copy()

print('Baidu rows after keeping first row per cid:', f'{len(baidu_unique):,}')
print('unique matched cid after dedup:', f"{baidu_unique['cid_int'].nunique():,}")

baidu_unique.head()


## 8. 和 full_data 2017 企业左连接

这里用左连接，所以不会引入 BaiduNetdisk 匹配桥里的额外企业。最终分母仍然是 `full_data.dta` 中 2017 年唯一企业数。


In [ ]:
full2017['cid_int'] = pd.to_numeric(full2017['cid'], errors='coerce')
full2017['cid_int'] = full2017['cid_int'].astype('Int64')

baidu_unique['cid_int'] = baidu_unique['cid_int'].astype('int64')

full_baidu_match = full2017.merge(
    baidu_unique,
    on='cid_int',
    how='left',
    suffixes=('', '_baidu'),
    indicator='baidu_merge',
)

full_baidu_match['baidu_matched'] = full_baidu_match['baidu_merge'] == 'both'

n_baidu_matched = int(full_baidu_match['baidu_matched'].sum())

print('full_data 2017 unique firms:', f'{n_full:,}')
print('matched using BaiduNetdisk bridge:', f'{n_baidu_matched:,}')
print('match rate using BaiduNetdisk bridge:', f'{n_baidu_matched / n_full:.2%}')

full_baidu_match['baidu_merge'].value_counts(dropna=False)


In [ ]:
full_baidu_match.head()


## 9. 汇总匹配情况

这里直接和之前当前项目桥接表口径的结果对比。之前口径结果是：

- full_data 2017 唯一企业：5,719,292；
- 当前 `cid_entid_unique.dta` + `H:\\汇算数据\\2017.dta` 最终匹配：1,594,239；
- 匹配率：27.87%。


In [ ]:
old_matched = 1_594_239

summary = pd.DataFrame([
    ['full_data 2017 unique firms', n_full, n_full, n_full / n_full],
    ['old bridge: cid_entid_unique + huisuan 2017', old_matched, n_full, old_matched / n_full],
    ['BaiduNetdisk bridge: final_joinby_matched_data_2017_With_cid', n_baidu_matched, n_full, n_baidu_matched / n_full],
], columns=['step', 'count', 'denominator', 'rate'])

summary['rate_percent'] = summary['rate'].map(lambda x: f'{x:.2%}')
summary


## 10. 可选：看 BaiduNetdisk 桥里的变量覆盖

如果 BaiduNetdisk 桥里有 `id` 或 `eid`，这里可以看匹配成功企业里这些变量是否缺失。


In [ ]:
matched_only = full_baidu_match.loc[full_baidu_match['baidu_matched']].copy()

print('matched rows:', f'{len(matched_only):,}')

for col in ['id', 'eid', 'match_step', 'company_name']:
    if col in matched_only.columns:
        print(col, 'nonmissing:', f"{matched_only[col].notna().sum():,}")

matched_only.head()


## 11. 可选：保存结果

如果只是想看匹配率，可以不运行这一步。

如果要保存完整检查结果，可以运行下面的 cell。


In [ ]:
save_cols = ['firm_id', 'cid', 'cid_int', 'baidu_matched']

for col in ['id', 'eid', 'match_step', 'company_name']:
    if col in full_baidu_match.columns:
        save_cols.append(col)

full_baidu_match[save_cols].to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print('saved CSV:', OUT_CSV)

# 如果需要 Stata dta 文件，再取消下面两行注释。
# full_baidu_match[save_cols].to_stata(OUT_DTA, write_index=False, version=118)
# print('saved DTA:', OUT_DTA)
